# Tron CSV dataset analysis
Install notebook dependencies with `%pip install "pandas>=2,<4" "networkx>=3,<4" "pyarrow>=14"`.
Set DRIVE_ROOT to the synced dataset folder. Local folders work without mounting Drive.


In [ ]:
from pathlib import Path
import pandas as pd


def colab_load_dataset(drive_root, compressed=False):
    """Mount Colab Drive when needed; load five append logs as string DataFrames.

    Local paths work without Colab. compressed=True prefers gzip, then Parquet
    for each table; False requires raw CSV. No type inference or deduplication
    occurs here: addresses, large integers, leading zeros and blanks are kept.
    """
    if type(compressed) is not bool:
        raise ValueError("compressed must be a boolean")
    root = Path(drive_root).expanduser().resolve()
    mountpoint = Path("/content/drive")
    if root.is_relative_to(mountpoint) and not (mountpoint / "MyDrive").is_dir():
        try:
            from google.colab import drive
        except ImportError as error:
            raise RuntimeError("This Drive path requires Google Colab; use a local dataset path outside Colab.") from error
        drive.mount(str(mountpoint))
    headers = {
        "seed_wallets.csv": [
            "wallet_address",
            "hex_address",
            "transaction_id",
            "source_name",
            "label",
            "created_at"
        ],
        "accounts.csv": [
            "address",
            "hex_address",
            "first_seen_timestamp",
            "last_seen_timestamp",
            "created_date",
            "created_at"
        ],
        "transactions.csv": [
            "tx_hash",
            "block_number",
            "block_timestamp",
            "from_address",
            "to_address",
            "token_symbol",
            "contract_address",
            "amount",
            "fee_trx",
            "created_at"
        ],
        "wallet_hops.csv": [
            "hop_id",
            "wallet_address",
            "seed_address",
            "parent_address",
            "via_tx_hash",
            "hop_level",
            "direction",
            "discovered_at"
        ],
        "pipeline_crawl_state.csv": [
            "address",
            "seed_address",
            "hop_level",
            "synced",
            "expanded",
            "is_supernode",
            "created_at",
            "updated_at"
        ]
    }
    dfs = {}
    for filename, columns in headers.items():
        stem = Path(filename).stem
        names = [filename + ".gz", stem + ".parquet"] if compressed else [filename]
        path = next((root / name for name in names if (root / name).is_file()), None)
        if path is None:
            raise FileNotFoundError(f"Missing dataset table {stem}; expected {' or '.join(names)} in {root}")
        if path.suffix == ".parquet":
            frame = pd.read_parquet(path).fillna("").astype(str)
        else:
            frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        if list(frame.columns) != columns:
            raise ValueError(f"Unexpected schema in {path.name}; expected {columns}")
        dfs[stem] = frame
    return dfs


In [ ]:
import networkx as nx


def colab_run_analysis(dfs) -> dict:
    """Rebuild the transaction graph and CSVStore's default health metrics.

    Append-log semantics match CSVStore.load(): latest seeds/accounts/state,
    first transaction per hash, first hop per (wallet, seed, level), label-0
    seeds only. Hop progress comes from crawl state, not shortest graph paths:
    pruned frontiers and incomplete crawls need not reflect graph distances.
    """
    required = {
        "seed_wallets": ["wallet_address", "label"],
        "accounts": ["address"],
        "transactions": ["tx_hash", "from_address", "to_address"],
        "wallet_hops": ["hop_id", "wallet_address", "seed_address", "hop_level"],
        "pipeline_crawl_state": ["address", "seed_address", "hop_level", "synced", "expanded", "is_supernode"],
    }
    rows = {}
    for name, columns in required.items():
        if name not in dfs or not set(columns).issubset(dfs[name].columns):
            raise ValueError(f"Missing table or required columns: {name}")
        rows[name] = dfs[name].fillna("").astype(str).to_dict("records")
    seeds = {row["wallet_address"]: row for row in rows["seed_wallets"] if row["wallet_address"]}
    seeds = {address: row for address, row in seeds.items() if row["label"] == "0"}
    accounts = {row["address"]: row for row in rows["accounts"] if row["address"]}
    transactions = {}
    for row in rows["transactions"]:
        if row["tx_hash"]:
            transactions.setdefault(row["tx_hash"], row)
    hops = {}
    for row in rows["wallet_hops"]:
        level = int(row["hop_level"])
        if int(row["hop_id"]) < 0 or level < 0 or not row["wallet_address"] or not row["seed_address"]:
            raise ValueError("Hop identifiers/levels must be nonnegative and wallet/seed required")
        hops.setdefault((row["wallet_address"], row["seed_address"], level), row)
    states = {}
    for row in rows["pipeline_crawl_state"]:
        level = int(row["hop_level"])
        if level < 0 or not row["address"]:
            raise ValueError("Crawl-state level must be nonnegative and address required")
        row["hop_level"] = level
        for field in ("synced", "expanded", "is_supernode"):
            encoded = row[field].strip().upper()
            if encoded not in {"TRUE", "FALSE"}:
                raise ValueError(f"Crawl-state {field} must be TRUE or FALSE")
            row[field] = encoded == "TRUE"
        states[row["address"]] = row
    pending_crawls = sum(not row["synced"] for row in states.values())
    pending_expands = sum(row["synced"] and not row["expanded"] for row in states.values())
    finished_seeds = 0
    for seed in seeds:
        descendants = [row for row in states.values() if row["seed_address"] == seed]
        if descendants and all(row["synced"] and row["expanded"] for row in descendants):
            finished_seeds += 1
    metrics = {
        "total_seeds": len(seeds),
        "total_accounts": len(accounts),
        "total_transactions": len(transactions),
        "total_hops": len(hops),
        "total_crawl_states": len(states),
        "supernodes": sum(row["is_supernode"] for row in states.values()),
        "pending_crawls": pending_crawls,
        "pending_expands": pending_expands,
        "per_hop": {level: sum(row["hop_level"] == level for row in states.values()) for level in range(6)},
        "finished_seeds": finished_seeds,
        "is_complete": bool(states) and pending_crawls == 0 and pending_expands == 0,
    }
    graph = nx.MultiDiGraph()
    graph.add_nodes_from(accounts)
    graph.add_nodes_from(seeds)
    graph.add_nodes_from((address, row) for address, row in states.items())
    for tx_hash, row in transactions.items():
        sender, receiver = row["from_address"], row["to_address"]
        if sender and receiver:
            graph.add_edge(sender, receiver, key=tx_hash, **row)
    return {
        "metrics": metrics,
        "graph": graph,
        "summary": {
            "nodes": graph.number_of_nodes(),
            "edges": graph.number_of_edges(),
            "weak_components": nx.number_weakly_connected_components(graph),
        },
    }


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pipeline_data"
dfs = colab_load_dataset(DRIVE_ROOT, compressed=False)
{name: len(frame) for name, frame in dfs.items()}
analysis = colab_run_analysis(dfs)
analysis["metrics"], analysis["summary"]
